# MiniDance — 上传数据集到 Hugging Face

把网盘里的 latent 分片和短字幕上传到 `JuniverseLi/minidance-scenery`（约 9 GB）。

- 运行时选 **CPU** 就行（不需要 GPU，几乎不花计算单元）。
- 需要一个有 **write** 权限的 HF token：在 https://huggingface.co/settings/tokens 新建，然后在 Colab 左侧 🔑「Secrets」里添加名为 `HF_TOKEN` 的密钥并打开 notebook 访问权限。没有的话运行时会让你粘贴。
- 上传可以断点续传：断开后重新运行全部单元格，已经传完的文件会自动跳过。


## 1. 挂载网盘，检查文件


In [ ]:
import os
DRIVE_DIR = '/content/drive/MyDrive/minidance'
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
shards = sorted(f for f in os.listdir(f'{DRIVE_DIR}/latents') if f.startswith('shard_'))
print(len(shards), 'shard files (应为 172 = 86 × 2)')
print('short_captions.jsonl:', os.path.exists(f'{DRIVE_DIR}/short_captions.jsonl'))


## 2. 登录 Hugging Face


In [ ]:
!pip install -q -U huggingface_hub
from huggingface_hub import HfApi, login
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    from getpass import getpass
    token = getpass('HF write token: ')
login(token=token)
api = HfApi()
print('logged in as', api.whoami()['name'])


## 3. 上传（约 9 GB，断开后重新运行即可续传）


In [ ]:
import glob, huggingface_hub
REPO = 'JuniverseLi/minidance-scenery'
PATTERNS = ['latents/shard_*.safetensors', 'latents/shard_*.parquet', 'short_captions.jsonl']
api.create_repo(REPO, repo_type='dataset', exist_ok=True)
print('huggingface_hub', huggingface_hub.__version__)

if hasattr(api, 'upload_large_folder'):
    api.upload_large_folder(repo_id=REPO, repo_type='dataset', folder_path=DRIVE_DIR, allow_patterns=PATTERNS)
else:
    # older huggingface_hub (Colab's preinstalled one until the session restarts): upload in small commits,
    # skipping files already on the Hub, so a rerun continues where it stopped
    done = set(api.list_repo_files(REPO, repo_type='dataset'))
    todo = sorted(os.path.relpath(f, DRIVE_DIR).replace(os.sep, '/')
                  for pat in PATTERNS for f in glob.glob(f'{DRIVE_DIR}/{pat}'))
    todo = [f for f in todo if f not in done]
    print(len(done), 'files already uploaded,', len(todo), 'to go')
    for i in range(0, len(todo), 16):
        batch = todo[i:i + 16]
        api.upload_folder(repo_id=REPO, repo_type='dataset', folder_path=DRIVE_DIR, allow_patterns=batch,
                          commit_message=f'Add {batch[0]} .. {batch[-1]}')
        print(f'{min(i + 16, len(todo))}/{len(todo)} uploaded', flush=True)
print(f'https://huggingface.co/datasets/{REPO}')
